In [1]:
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns

import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
from sklearn.model_selection import train_test_split
from sksurv.ensemble import RandomSurvivalForest
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored , concordance_index_ipcw
from sklearn.impute import SimpleImputer
from sksurv.util import Surv

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor




In [2]:
# Molecular Data
mol_df = pl.read_csv("../data/raw/X_train/molecular_train.csv")
mol_eval = pl.read_csv("../data/raw/X_test/molecular_test.csv" , ignore_errors=True)

cl_df = pl.read_csv("../data/raw/X_train/clinical_train.csv")

y_train = pl.read_csv("../data/raw/target_train.csv")              

In [11]:
mol_df["REF"].value_counts().sort(by=pl.col("REF").str.len_chars() , descending=True)

REF,count
str,u32
null,114
"""ACAGGTGAGGCACCTGAGATTAGAGCAGGA…",1
"""TCCGTAGCCACCGCCCCCGTACCTGCGGGG…",1
"""CAGCGGCAACGCCTCGCTCATCTTGCCTGG…",2
"""AGCAGAGGCTTAAGGAGGAGGAAGAAGACA…",11
…,…
"""AT""",48
"""A""",1234
"""G""",2965


In [195]:
mol_df.filter(pl.col("PROTEIN_CHANGE").is_null())

# p.?

ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH
str,str,f64,f64,str,str,str,str,str,f64,f64
"""P100378""","""5""",1.295228e6,1.295228e6,"""G""","""A""","""TERT""",null,"""2KB_upstream_variant""",0.513,46.0
"""P102718""","""5""",1.295228e6,1.295228e6,"""G""","""A""","""TERT""",null,"""2KB_upstream_variant""",0.267,54.0
"""P105914""","""5""",1.295228e6,1.295228e6,"""G""","""A""","""TERT""",null,"""2KB_upstream_variant""",0.232,59.0
"""P120956""","""5""",1.295228e6,1.295228e6,"""G""","""A""","""TERT""",null,"""2KB_upstream_variant""",0.202,92.0
"""P120967""","""5""",1.295228e6,1.295228e6,"""G""","""A""","""TERT""",null,"""2KB_upstream_variant""",0.052,112.0
…,…,…,…,…,…,…,…,…,…,…
"""P121131""","""5""",1.295228e6,1.295228e6,"""G""","""A""","""TERT""",null,"""2KB_upstream_variant""",0.316,118.0
"""P131530""","""5""",1.295228e6,1.295228e6,"""G""","""A""","""TERT""",null,"""2KB_upstream_variant""",0.115,229.0
"""P131838""","""5""",1.295228e6,1.295228e6,"""G""","""A""","""TERT""",null,"""2KB_upstream_variant""",0.151,224.0


In [197]:


mol_df = mol_df.with_columns(
    pl.when(pl.col("PROTEIN_CHANGE").is_null())
    .then(pl.lit("p.?"))
    .otherwise(pl.col("PROTEIN_CHANGE"))
    .alias("PROTEIN_CHANGE")
)


In [201]:
mol_df.filter(pl.col("PROTEIN_CHANGE") == "p.?" , pl.col("GENE") == "TERT")

ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH
str,str,f64,f64,str,str,str,str,str,f64,f64
"""P100378""","""5""",1.295228e6,1.295228e6,"""G""","""A""","""TERT""","""p.?""","""2KB_upstream_variant""",0.513,46.0
"""P102718""","""5""",1.295228e6,1.295228e6,"""G""","""A""","""TERT""","""p.?""","""2KB_upstream_variant""",0.267,54.0
"""P105914""","""5""",1.295228e6,1.295228e6,"""G""","""A""","""TERT""","""p.?""","""2KB_upstream_variant""",0.232,59.0
"""P120956""","""5""",1.295228e6,1.295228e6,"""G""","""A""","""TERT""","""p.?""","""2KB_upstream_variant""",0.202,92.0
"""P120967""","""5""",1.295228e6,1.295228e6,"""G""","""A""","""TERT""","""p.?""","""2KB_upstream_variant""",0.052,112.0
…,…,…,…,…,…,…,…,…,…,…
"""P121131""","""5""",1.295228e6,1.295228e6,"""G""","""A""","""TERT""","""p.?""","""2KB_upstream_variant""",0.316,118.0
"""P131530""","""5""",1.295228e6,1.295228e6,"""G""","""A""","""TERT""","""p.?""","""2KB_upstream_variant""",0.115,229.0
"""P131838""","""5""",1.295228e6,1.295228e6,"""G""","""A""","""TERT""","""p.?""","""2KB_upstream_variant""",0.151,224.0


In [90]:
mol_df.filter(pl.col("REF").str.len_chars() > 1 , pl.col("ALT").str.len_chars() > 1) 

# is_complex_nucleotide


def is_complex_nucleotide(ref , alt):
    if(isinstance(ref,str) and isinstance(alt,str)):
        return int(len(ref) > 1 and len(alt) > 1)


mol_df = mol_df.with_columns(
    pl.struct(["REF","ALT"]).map_elements(lambda row : is_complex_nucleotide(row["REF"] , row["ALT"])
                                          , return_dtype=pl.Int16 ).alias("is_complex_nucleotide")
)

In [91]:
mol_df = mol_df.with_columns(
    pl.col("REF").str.len_chars().alias("REF_length")
)

In [92]:
mol_df = mol_df.with_columns(
    pl.col("ALT").str.len_chars().alias("ALT_length")
)

In [93]:
mol_df.sort(pl.col("REF_length") , descending=True , nulls_last=True)

ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH,is_complex_nucleotide,REF_length,ALT_length
str,str,f64,f64,str,str,str,str,str,f64,f64,i16,u32,u32
"""P117988""","""2""",2.5966666e7,2.5967241e7,"""ACAGGTGAGGCACCTGAGATTAGAGCAGGA…","""A""","""ASXL2""","""p.R655fs*6""","""frameshift_variant""",0.2689,1259.0,0,576,1
"""P118264""","""17""",7.4732895e7,7.4732964e7,"""TCCGTAGCCACCGCCCCCGTACCTGCGGGG…","""T""","""SRSF2""","""p.P95_R117delPPDSHHSRRGPPPRRYG…","""inframe_codon_loss""",0.1228,1422.0,0,70,1
"""P102939""","""21""",3.6259317e7,3.6259372e7,"""CAGCGGCAACGCCTCGCTCATCTTGCCTGG…","""C""","""RUNX1""","""p.F40fs*14""","""frameshift_variant""",0.0307,1536.0,0,56,1
"""P105849""","""21""",3.6259317e7,3.6259372e7,"""CAGCGGCAACGCCTCGCTCATCTTGCCTGG…","""C""","""RUNX1""","""p.F40fs*14""","""frameshift_variant""",0.3117,1147.0,0,56,1
"""P102638""","""19""",1.3054564e7,1.3054616e7,"""AGCAGAGGCTTAAGGAGGAGGAAGAAGACA…","""A""","""CALR""","""p.L367fs*46""","""frameshift_variant""",0.2546,1720.0,0,53,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""P131472""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",null,null,null,null,null
"""P131505""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",null,null,null,null,null
"""P131816""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",null,null,null,null,null


In [94]:
# Transitions

# purine : A <-> G 
# Pyrimidine : C <-> T


def is_transition(str1 , str2):
    # Purine
    if(isinstance(str1,str) & isinstance(str2,str)):
        if( (str1=="A" and str2 == "G") | (str1 == "G") and (str2 == "A")):
            return 1
        elif((str1=="C" and str2 == "T") | (str1 == "T") and (str2 == "C")):
            return 1
        else:
            return 0
    


mol_df = mol_df.with_columns(
    pl.struct(["REF", "ALT"]).map_elements(
        lambda row: is_transition(row["REF"], row["ALT"]) , return_dtype=pl.Int32
    ).alias("is_transition")
)


In [95]:
def is_transversion(str1 , str2):
    nucleotide = (str1,str2)
    trans_vers = [("A" ,"C") , ("C","A") , ("C" , "G") , ("G" , "C") , ("G" , "T" ) , ("T" ,  "G") , ("A" , "T") , ("T" , "A")]
    
    
    return int(nucleotide in trans_vers)


mol_df = mol_df.with_columns(
    pl.struct(["REF", "ALT"]).map_elements(
        lambda row: is_transversion(row["REF"], row["ALT"]) , return_dtype=pl.Int32
    ).alias("is_transversion")
)



In [96]:
# Insertion - deletion

def is_indel(str1 , str2):
    if(isinstance(str1,str) & isinstance(str2,str)):
        return int(len(str1) != len(str2))
    

mol_df = mol_df.with_columns(
    pl.struct(["REF", "ALT"]).map_elements(
        lambda row: is_indel(row["REF"], row["ALT"]) , return_dtype=pl.Int32
    ).alias("is_indel")
)

In [97]:
mol_df.filter(pl.col("REF").is_null())

ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH,is_complex_nucleotide,REF_length,ALT_length,is_transition,is_transversion,is_indel
str,str,f64,f64,str,str,str,str,str,f64,f64,i16,u32,u32,i32,i32,i32
"""P100016""",null,null,null,null,null,"""FLT3""","""FLT3_ITD""","""ITD""",0.182681,null,null,null,null,null,0,null
"""P100022""",null,null,null,null,null,"""FLT3""","""FLT3_ITD""","""ITD""",0.130882,null,null,null,null,null,0,null
"""P100123""",null,null,null,null,null,"""FLT3""","""FLT3_ITD""","""ITD""",0.265306,null,null,null,null,null,0,null
"""P100348""",null,null,null,null,null,"""FLT3""","""FLT3_ITD""","""ITD""",0.066667,null,null,null,null,null,0,null
"""P102822""",null,null,null,null,null,"""FLT3""","""FLT3_ITD""","""ITD""",0.049267,null,null,null,null,null,0,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""P131472""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",null,null,null,null,null,null,0,null
"""P131505""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",null,null,null,null,null,null,0,null
"""P131816""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",null,null,null,null,null,null,0,null


# TRAITEMENT DE MOLECULAR DATA SET


In [98]:
df_mol_cl = mol_df.join(cl_df , on="ID" , how="left")

df_mol_cl

ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH,is_complex_nucleotide,REF_length,ALT_length,is_transition,is_transversion,is_indel,CENTER,BM_BLAST,WBC,ANC,MONOCYTES,HB,PLT,CYTOGENETICS
str,str,f64,f64,str,str,str,str,str,f64,f64,i16,u32,u32,i32,i32,i32,str,f64,f64,f64,f64,f64,f64,str
"""P100000""","""11""",1.19149248e8,1.19149248e8,"""G""","""A""","""CBL""","""p.C419Y""","""non_synonymous_codon""",0.083,1308.0,0,1,1,1,0,0,"""IHBT""",3.0,7.5,4.65,null,8.8,406.0,"""46,xx,del(5)(q15q33.3)[3]/46,x…"
"""P100000""","""5""",1.31822301e8,1.31822301e8,"""G""","""T""","""IRF1""","""p.Y164*""","""stop_gained""",0.022,532.0,0,1,1,0,1,0,"""IHBT""",3.0,7.5,4.65,null,8.8,406.0,"""46,xx,del(5)(q15q33.3)[3]/46,x…"
"""P100000""","""3""",7.769406e7,7.769406e7,"""G""","""C""","""ROBO2""","""p.?""","""splice_site_variant""",0.41,876.0,0,1,1,0,1,0,"""IHBT""",3.0,7.5,4.65,null,8.8,406.0,"""46,xx,del(5)(q15q33.3)[3]/46,x…"
"""P100000""","""4""",1.06164917e8,1.06164917e8,"""G""","""T""","""TET2""","""p.R1262L""","""non_synonymous_codon""",0.43,826.0,0,1,1,0,1,0,"""IHBT""",3.0,7.5,4.65,null,8.8,406.0,"""46,xx,del(5)(q15q33.3)[3]/46,x…"
"""P100000""","""2""",2.5468147e7,2.5468163e7,"""ACGAAGAGGGGGTGTTC""","""A""","""DNMT3A""","""p.E505fs*141""","""frameshift_variant""",0.0898,942.0,0,17,1,0,0,1,"""IHBT""",3.0,7.5,4.65,null,8.8,406.0,"""46,xx,del(5)(q15q33.3)[3]/46,x…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""P131472""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",null,null,null,null,null,null,0,null,"""TUD""",17.5,8.57,5.9,0.65,11.52,16.0,"""46,xy,del(20)(q)"""
"""P131505""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",null,null,null,null,null,null,0,null,"""TUD""",8.0,3.2,null,null,10.34,66.0,"""45,x,-y/46,xy"""
"""P131816""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",null,null,null,null,null,null,0,null,"""KI""",18.0,34.2,4.8,19.4,8.0,155.0,"""46,xx[20]"""


In [99]:
df_mol_cl = df_mol_cl.join(y_train , on="ID"  , how="left")

df_mol_cl

ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH,is_complex_nucleotide,REF_length,ALT_length,is_transition,is_transversion,is_indel,CENTER,BM_BLAST,WBC,ANC,MONOCYTES,HB,PLT,CYTOGENETICS,OS_YEARS,OS_STATUS
str,str,f64,f64,str,str,str,str,str,f64,f64,i16,u32,u32,i32,i32,i32,str,f64,f64,f64,f64,f64,f64,str,f64,f64
"""P100000""","""11""",1.19149248e8,1.19149248e8,"""G""","""A""","""CBL""","""p.C419Y""","""non_synonymous_codon""",0.083,1308.0,0,1,1,1,0,0,"""IHBT""",3.0,7.5,4.65,null,8.8,406.0,"""46,xx,del(5)(q15q33.3)[3]/46,x…",5.819178,0.0
"""P100000""","""5""",1.31822301e8,1.31822301e8,"""G""","""T""","""IRF1""","""p.Y164*""","""stop_gained""",0.022,532.0,0,1,1,0,1,0,"""IHBT""",3.0,7.5,4.65,null,8.8,406.0,"""46,xx,del(5)(q15q33.3)[3]/46,x…",5.819178,0.0
"""P100000""","""3""",7.769406e7,7.769406e7,"""G""","""C""","""ROBO2""","""p.?""","""splice_site_variant""",0.41,876.0,0,1,1,0,1,0,"""IHBT""",3.0,7.5,4.65,null,8.8,406.0,"""46,xx,del(5)(q15q33.3)[3]/46,x…",5.819178,0.0
"""P100000""","""4""",1.06164917e8,1.06164917e8,"""G""","""T""","""TET2""","""p.R1262L""","""non_synonymous_codon""",0.43,826.0,0,1,1,0,1,0,"""IHBT""",3.0,7.5,4.65,null,8.8,406.0,"""46,xx,del(5)(q15q33.3)[3]/46,x…",5.819178,0.0
"""P100000""","""2""",2.5468147e7,2.5468163e7,"""ACGAAGAGGGGGTGTTC""","""A""","""DNMT3A""","""p.E505fs*141""","""frameshift_variant""",0.0898,942.0,0,17,1,0,0,1,"""IHBT""",3.0,7.5,4.65,null,8.8,406.0,"""46,xx,del(5)(q15q33.3)[3]/46,x…",5.819178,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""P131472""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",null,null,null,null,null,null,0,null,"""TUD""",17.5,8.57,5.9,0.65,11.52,16.0,"""46,xy,del(20)(q)""",0.671233,0.0
"""P131505""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",null,null,null,null,null,null,0,null,"""TUD""",8.0,3.2,null,null,10.34,66.0,"""45,x,-y/46,xy""",3.69589,0.0
"""P131816""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",null,null,null,null,null,null,0,null,"""KI""",18.0,34.2,4.8,19.4,8.0,155.0,"""46,xx[20]""",0.468493,1.0


In [100]:
def is_snv_REF(df , col , letter):
    df = df.with_columns(
        pl.when((pl.col(col) == letter))
        .then(1)
        .otherwise(0)
        .alias(f"{col}_{letter}")
    )
    
    return df

snv_REF  = ['A','C','G','T']

for snv in snv_REF:
    mol_df =  is_snv_REF(mol_df , "REF" , snv)
    
    
mol_df

ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH,is_complex_nucleotide,REF_length,ALT_length,is_transition,is_transversion,is_indel,REF_A,REF_C,REF_G,REF_T
str,str,f64,f64,str,str,str,str,str,f64,f64,i16,u32,u32,i32,i32,i32,i32,i32,i32,i32
"""P100000""","""11""",1.19149248e8,1.19149248e8,"""G""","""A""","""CBL""","""p.C419Y""","""non_synonymous_codon""",0.083,1308.0,0,1,1,1,0,0,0,0,1,0
"""P100000""","""5""",1.31822301e8,1.31822301e8,"""G""","""T""","""IRF1""","""p.Y164*""","""stop_gained""",0.022,532.0,0,1,1,0,1,0,0,0,1,0
"""P100000""","""3""",7.769406e7,7.769406e7,"""G""","""C""","""ROBO2""","""p.?""","""splice_site_variant""",0.41,876.0,0,1,1,0,1,0,0,0,1,0
"""P100000""","""4""",1.06164917e8,1.06164917e8,"""G""","""T""","""TET2""","""p.R1262L""","""non_synonymous_codon""",0.43,826.0,0,1,1,0,1,0,0,0,1,0
"""P100000""","""2""",2.5468147e7,2.5468163e7,"""ACGAAGAGGGGGTGTTC""","""A""","""DNMT3A""","""p.E505fs*141""","""frameshift_variant""",0.0898,942.0,0,17,1,0,0,1,0,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""P131472""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",null,null,null,null,null,null,0,null,0,0,0,0
"""P131505""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",null,null,null,null,null,null,0,null,0,0,0,0
"""P131816""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",null,null,null,null,null,null,0,null,0,0,0,0


In [101]:
for snv in snv_REF:
    mol_df = is_snv_REF(mol_df , "ALT" , snv)
    
mol_df

ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH,is_complex_nucleotide,REF_length,ALT_length,is_transition,is_transversion,is_indel,REF_A,REF_C,REF_G,REF_T,ALT_A,ALT_C,ALT_G,ALT_T
str,str,f64,f64,str,str,str,str,str,f64,f64,i16,u32,u32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32
"""P100000""","""11""",1.19149248e8,1.19149248e8,"""G""","""A""","""CBL""","""p.C419Y""","""non_synonymous_codon""",0.083,1308.0,0,1,1,1,0,0,0,0,1,0,1,0,0,0
"""P100000""","""5""",1.31822301e8,1.31822301e8,"""G""","""T""","""IRF1""","""p.Y164*""","""stop_gained""",0.022,532.0,0,1,1,0,1,0,0,0,1,0,0,0,0,1
"""P100000""","""3""",7.769406e7,7.769406e7,"""G""","""C""","""ROBO2""","""p.?""","""splice_site_variant""",0.41,876.0,0,1,1,0,1,0,0,0,1,0,0,1,0,0
"""P100000""","""4""",1.06164917e8,1.06164917e8,"""G""","""T""","""TET2""","""p.R1262L""","""non_synonymous_codon""",0.43,826.0,0,1,1,0,1,0,0,0,1,0,0,0,0,1
"""P100000""","""2""",2.5468147e7,2.5468163e7,"""ACGAAGAGGGGGTGTTC""","""A""","""DNMT3A""","""p.E505fs*141""","""frameshift_variant""",0.0898,942.0,0,17,1,0,0,1,0,0,0,0,1,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""P131472""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",null,null,null,null,null,null,0,null,0,0,0,0,0,0,0,0
"""P131505""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",null,null,null,null,null,null,0,null,0,0,0,0,0,0,0,0
"""P131816""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",null,null,null,null,null,null,0,null,0,0,0,0,0,0,0,0


In [102]:
def complex_nucleotide(df , col ):
    df = df.with_columns(
        pl.when(pl.col(col).str.len_chars() > 1)
        .then(1)
        .otherwise(0)
        .alias(f"complex_nucleotide_{col}")
    )
    
    return df

mol_df = complex_nucleotide(mol_df , "REF")
mol_df = complex_nucleotide(mol_df , "ALT")



In [103]:
mol_df

ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH,is_complex_nucleotide,REF_length,ALT_length,is_transition,is_transversion,is_indel,REF_A,REF_C,REF_G,REF_T,ALT_A,ALT_C,ALT_G,ALT_T,complex_nucleotide_REF,complex_nucleotide_ALT
str,str,f64,f64,str,str,str,str,str,f64,f64,i16,u32,u32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32
"""P100000""","""11""",1.19149248e8,1.19149248e8,"""G""","""A""","""CBL""","""p.C419Y""","""non_synonymous_codon""",0.083,1308.0,0,1,1,1,0,0,0,0,1,0,1,0,0,0,0,0
"""P100000""","""5""",1.31822301e8,1.31822301e8,"""G""","""T""","""IRF1""","""p.Y164*""","""stop_gained""",0.022,532.0,0,1,1,0,1,0,0,0,1,0,0,0,0,1,0,0
"""P100000""","""3""",7.769406e7,7.769406e7,"""G""","""C""","""ROBO2""","""p.?""","""splice_site_variant""",0.41,876.0,0,1,1,0,1,0,0,0,1,0,0,1,0,0,0,0
"""P100000""","""4""",1.06164917e8,1.06164917e8,"""G""","""T""","""TET2""","""p.R1262L""","""non_synonymous_codon""",0.43,826.0,0,1,1,0,1,0,0,0,1,0,0,0,0,1,0,0
"""P100000""","""2""",2.5468147e7,2.5468163e7,"""ACGAAGAGGGGGTGTTC""","""A""","""DNMT3A""","""p.E505fs*141""","""frameshift_variant""",0.0898,942.0,0,17,1,0,0,1,0,0,0,0,1,0,0,0,1,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""P131472""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",null,null,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0
"""P131505""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",null,null,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0
"""P131816""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",null,null,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0


In [104]:
lst_ref = mol_df.filter(pl.col("REF").str.len_chars() > 1)["REF"].unique().to_list()

for ref in lst_ref:
    for ch in ref:
        if ch not in ['A','C','G','T']:
            print(ref)

In [105]:
def is_SNV(df , col):
    df = df.with_columns(
        pl.when(pl.col(col).str.len_chars() == 1)
        .then(1)
        .otherwise(0)
        .alias(f"{col} is_SNV")
    )
    
    return df

mol_df = is_SNV(mol_df,"REF")

mol_df

ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH,is_complex_nucleotide,REF_length,ALT_length,is_transition,is_transversion,is_indel,REF_A,REF_C,REF_G,REF_T,ALT_A,ALT_C,ALT_G,ALT_T,complex_nucleotide_REF,complex_nucleotide_ALT,REF is_SNV
str,str,f64,f64,str,str,str,str,str,f64,f64,i16,u32,u32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32
"""P100000""","""11""",1.19149248e8,1.19149248e8,"""G""","""A""","""CBL""","""p.C419Y""","""non_synonymous_codon""",0.083,1308.0,0,1,1,1,0,0,0,0,1,0,1,0,0,0,0,0,1
"""P100000""","""5""",1.31822301e8,1.31822301e8,"""G""","""T""","""IRF1""","""p.Y164*""","""stop_gained""",0.022,532.0,0,1,1,0,1,0,0,0,1,0,0,0,0,1,0,0,1
"""P100000""","""3""",7.769406e7,7.769406e7,"""G""","""C""","""ROBO2""","""p.?""","""splice_site_variant""",0.41,876.0,0,1,1,0,1,0,0,0,1,0,0,1,0,0,0,0,1
"""P100000""","""4""",1.06164917e8,1.06164917e8,"""G""","""T""","""TET2""","""p.R1262L""","""non_synonymous_codon""",0.43,826.0,0,1,1,0,1,0,0,0,1,0,0,0,0,1,0,0,1
"""P100000""","""2""",2.5468147e7,2.5468163e7,"""ACGAAGAGGGGGTGTTC""","""A""","""DNMT3A""","""p.E505fs*141""","""frameshift_variant""",0.0898,942.0,0,17,1,0,0,1,0,0,0,0,1,0,0,0,1,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""P131472""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",null,null,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0
"""P131505""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",null,null,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0
"""P131816""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",null,null,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0


In [106]:
mol_df.group_by(["GENE", "EFFECT"]).agg(
        pl.len().alias("mutations_per_gene_effect")
    )

GENE,EFFECT,mutations_per_gene_effect
str,str,u32
"""EGFR""","""non_synonymous_codon""",10
"""CEBPA""","""non_synonymous_codon""",26
"""FLT3""","""non_synonymous_codon""",30
"""TP53""","""inframe_codon_gain""",2
"""ETV6""","""frameshift_variant""",32
…,…,…
"""ARID1A""","""splice_site_variant""",1
"""CALR""","""inframe_codon_gain""",1
"""SH2B3""","""splice_site_variant""",1


In [107]:
def add_mutation_density_features(mol_df):
    """Add mutation density features to the molecular dataframe"""

    # Count mutations per gene
    gene_counts = mol_df.group_by("GENE").agg(
        pl.len().alias("mutations_per_gene")
    )

    # Count mutations per gene and effect type
    gene_effect_counts = mol_df.group_by(["GENE", "EFFECT"]).agg(
        pl.len().alias("mutations_per_gene_effect")
    )

    # Count mutations per chromosome region (binned)
    region_counts = mol_df.group_by(["CHR", "START"]).agg(
        pl.count().alias("mutations_per_region")
    )

    # Join these counts back to the original dataframe
    mol_df = mol_df.join(gene_counts, on="GENE", how="left")

    # For gene-effect counts, we need to join on both columns
    mol_df = mol_df.join(gene_effect_counts, on=["GENE", "EFFECT"], how="left")
    
    # Join region counts
    mol_df = mol_df.join(region_counts, on=["CHR", "START"], how="left")

    return mol_df


mol_df = add_mutation_density_features(mol_df)

C:\Users\zakar\AppData\Local\Temp\ipykernel_16716\609172987.py:16: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
  pl.count().alias("mutations_per_region")


In [108]:
mol_df

ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH,is_complex_nucleotide,REF_length,ALT_length,is_transition,is_transversion,is_indel,REF_A,REF_C,REF_G,REF_T,ALT_A,ALT_C,ALT_G,ALT_T,complex_nucleotide_REF,complex_nucleotide_ALT,REF is_SNV,mutations_per_gene,mutations_per_gene_effect,mutations_per_region
str,str,f64,f64,str,str,str,str,str,f64,f64,i16,u32,u32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,u32,u32,u32
"""P100000""","""11""",1.19149248e8,1.19149248e8,"""G""","""A""","""CBL""","""p.C419Y""","""non_synonymous_codon""",0.083,1308.0,0,1,1,1,0,0,0,0,1,0,1,0,0,0,0,0,1,228,163,1
"""P100000""","""5""",1.31822301e8,1.31822301e8,"""G""","""T""","""IRF1""","""p.Y164*""","""stop_gained""",0.022,532.0,0,1,1,0,1,0,0,0,1,0,0,0,0,1,0,0,1,22,4,1
"""P100000""","""3""",7.769406e7,7.769406e7,"""G""","""C""","""ROBO2""","""p.?""","""splice_site_variant""",0.41,876.0,0,1,1,0,1,0,0,0,1,0,0,1,0,0,0,0,1,17,3,3
"""P100000""","""4""",1.06164917e8,1.06164917e8,"""G""","""T""","""TET2""","""p.R1262L""","""non_synonymous_codon""",0.43,826.0,0,1,1,0,1,0,0,0,1,0,0,0,0,1,0,0,1,1663,417,4
"""P100000""","""2""",2.5468147e7,2.5468163e7,"""ACGAAGAGGGGGTGTTC""","""A""","""DNMT3A""","""p.E505fs*141""","""frameshift_variant""",0.0898,942.0,0,17,1,0,0,1,0,0,0,0,1,0,0,0,1,0,0,604,93,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""P131472""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",null,null,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null
"""P131505""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",null,null,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null
"""P131816""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",null,null,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null


In [109]:
mol_df.describe()

statistic,ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH,is_complex_nucleotide,REF_length,ALT_length,is_transition,is_transversion,is_indel,REF_A,REF_C,REF_G,REF_T,ALT_A,ALT_C,ALT_G,ALT_T,complex_nucleotide_REF,complex_nucleotide_ALT,REF is_SNV,mutations_per_gene,mutations_per_gene_effect,mutations_per_region
str,str,str,f64,f64,str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""10935""","""10821""",10821.0,10821.0,"""10821""","""10821""","""10935""","""10923""","""10935""",10846.0,10821.0,10821.0,10821.0,10821.0,10821.0,10935.0,10821.0,10935.0,10935.0,10935.0,10935.0,10935.0,10935.0,10935.0,10935.0,10935.0,10935.0,10935.0,10935.0,10935.0,10821.0
"""null_count""","""0""","""114""",114.0,114.0,"""114""","""114""","""0""","""12""","""0""",89.0,114.0,114.0,114.0,114.0,114.0,0.0,114.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,114.0
"""mean""",null,null,8.0783e7,8.0783e7,null,null,null,null,null,0.305087,1051.229554,0.006376,1.969781,1.427594,0.232511,0.241518,0.29452,0.112849,0.292547,0.271148,0.161957,0.232373,0.166164,0.122176,0.319616,0.151075,0.149246,0.8385,576.027709,283.047371,65.785879
"""std""",null,null,5.6427e7,5.6427e7,null,null,null,null,null,0.211524,552.861902,0.079602,6.879294,2.616076,0.422452,0.428023,0.455848,0.316422,0.454953,0.444572,0.368428,0.422365,0.372245,0.327505,0.466349,0.358138,0.356347,0.368008,539.530171,264.147822,138.89073
"""min""","""P100000""","""1""",394899.0,394899.0,"""A""","""A""","""ABL1""","""FLT3_ITD""","""2KB_upstream_variant""",0.02,16.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
"""25%""",null,null,3.1022441e7,3.1022442e7,null,null,null,null,null,0.1025,660.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,149.0,51.0,1.0
"""50%""",null,null,7.4732959e7,7.4732959e7,null,null,null,null,null,0.3213,975.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,487.0,166.0,3.0
"""75%""",null,null,1.15258747e8,1.15258747e8,null,null,null,null,null,0.442,1353.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,775.0,449.0,33.0
"""max""","""P132729""","""X""",2.26252135e8,2.26252135e8,"""TTTTCA""","""TTTTC""","""ZRSR2""","""p.Y974fs*8""","""synonymous_codon""",0.999,7156.0,1.0,576.0,76.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1663.0,773.0,485.0


In [110]:
mol_df.null_count()

ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH,is_complex_nucleotide,REF_length,ALT_length,is_transition,is_transversion,is_indel,REF_A,REF_C,REF_G,REF_T,ALT_A,ALT_C,ALT_G,ALT_T,complex_nucleotide_REF,complex_nucleotide_ALT,REF is_SNV,mutations_per_gene,mutations_per_gene_effect,mutations_per_region
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,114,114,114,114,114,0,12,0,89,114,114,114,114,114,0,114,0,0,0,0,0,0,0,0,0,0,0,0,0,114


In [111]:
def iqr_method(df , cl_lst):

    for (index , cl )  in enumerate(cl_lst):
        q1 = df.select(pl.col(cl).quantile(0.25)).item()
        q3 = df.select(pl.col(cl).quantile(0.75)).item()
        
        iqr = q3 - q1
        
        
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        
        # Winsorisation : capped values
        df = df.with_columns(
            pl.when(pl.col(cl) < lower_bound).then(lower_bound)
            .when(pl.col(cl) > upper_bound).then(upper_bound)
            .otherwise(pl.col(cl))
            .alias(cl)
        )
        
        
        
    return df

mol_df = iqr_method(mol_df , ["VAF" , "DEPTH" , "START" , "END"])



In [112]:
mol_df.describe()

statistic,ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH,is_complex_nucleotide,REF_length,ALT_length,is_transition,is_transversion,is_indel,REF_A,REF_C,REF_G,REF_T,ALT_A,ALT_C,ALT_G,ALT_T,complex_nucleotide_REF,complex_nucleotide_ALT,REF is_SNV,mutations_per_gene,mutations_per_gene_effect,mutations_per_region
str,str,str,f64,f64,str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""10935""","""10821""",10821.0,10821.0,"""10821""","""10821""","""10935""","""10923""","""10935""",10846.0,10821.0,10821.0,10821.0,10821.0,10821.0,10935.0,10821.0,10935.0,10935.0,10935.0,10935.0,10935.0,10935.0,10935.0,10935.0,10935.0,10935.0,10935.0,10935.0,10935.0,10821.0
"""null_count""","""0""","""114""",114.0,114.0,"""114""","""114""","""0""","""12""","""0""",89.0,114.0,114.0,114.0,114.0,114.0,0.0,114.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,114.0
"""mean""",null,null,8.0783e7,8.0783e7,null,null,null,null,null,0.3049,1041.753627,0.006376,1.969781,1.427594,0.232511,0.241518,0.29452,0.112849,0.292547,0.271148,0.161957,0.232373,0.166164,0.122176,0.319616,0.151075,0.149246,0.8385,576.027709,283.047371,65.785879
"""std""",null,null,5.6427e7,5.6427e7,null,null,null,null,null,0.210937,516.987451,0.079602,6.879294,2.616076,0.422452,0.428023,0.455848,0.316422,0.454953,0.444572,0.368428,0.422365,0.372245,0.327505,0.466349,0.358138,0.356347,0.368008,539.530171,264.147822,138.89073
"""min""","""P100000""","""1""",394899.0,394899.0,"""A""","""A""","""ABL1""","""FLT3_ITD""","""2KB_upstream_variant""",0.02,16.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
"""25%""",null,null,3.1022441e7,3.1022442e7,null,null,null,null,null,0.1025,660.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,149.0,51.0,1.0
"""50%""",null,null,7.4732959e7,7.4732959e7,null,null,null,null,null,0.3213,975.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,487.0,166.0,3.0
"""75%""",null,null,1.15258747e8,1.15258747e8,null,null,null,null,null,0.442,1353.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,775.0,449.0,33.0
"""max""","""P132729""","""X""",2.26252135e8,2.26252135e8,"""TTTTCA""","""TTTTC""","""ZRSR2""","""p.Y974fs*8""","""synonymous_codon""",0.95125,2392.5,1.0,576.0,76.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1663.0,773.0,485.0


# VAF , DEPTH TREATMENT (FAIT)

In [113]:
quant_vars = ["VAF" , "DEPTH"]
sub_df = mol_df.select(quant_vars)

# Conver to pandas
sub_pd = sub_df.to_pandas()
 
# Apply Model-Based Imputation (Random Forest)
imputer = IterativeImputer(estimator=RandomForestRegressor(), random_state=42)
imputed_data = imputer.fit_transform(sub_pd)

# 4. To Polars
imputed_pl = pl.DataFrame(imputed_data, schema=sub_df.columns)

# 5. Replace nulls
mol_df = mol_df.with_columns([imputed_pl[col].alias(col) for col in quant_vars])



c:\Users\zakar\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\impute\_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


In [114]:
mol_df

ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH,is_complex_nucleotide,REF_length,ALT_length,is_transition,is_transversion,is_indel,REF_A,REF_C,REF_G,REF_T,ALT_A,ALT_C,ALT_G,ALT_T,complex_nucleotide_REF,complex_nucleotide_ALT,REF is_SNV,mutations_per_gene,mutations_per_gene_effect,mutations_per_region
str,str,f64,f64,str,str,str,str,str,f64,f64,i16,u32,u32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,u32,u32,u32
"""P100000""","""11""",1.19149248e8,1.19149248e8,"""G""","""A""","""CBL""","""p.C419Y""","""non_synonymous_codon""",0.083,1308.0,0,1,1,1,0,0,0,0,1,0,1,0,0,0,0,0,1,228,163,1
"""P100000""","""5""",1.31822301e8,1.31822301e8,"""G""","""T""","""IRF1""","""p.Y164*""","""stop_gained""",0.022,532.0,0,1,1,0,1,0,0,0,1,0,0,0,0,1,0,0,1,22,4,1
"""P100000""","""3""",7.769406e7,7.769406e7,"""G""","""C""","""ROBO2""","""p.?""","""splice_site_variant""",0.41,876.0,0,1,1,0,1,0,0,0,1,0,0,1,0,0,0,0,1,17,3,3
"""P100000""","""4""",1.06164917e8,1.06164917e8,"""G""","""T""","""TET2""","""p.R1262L""","""non_synonymous_codon""",0.43,826.0,0,1,1,0,1,0,0,0,1,0,0,0,0,1,0,0,1,1663,417,4
"""P100000""","""2""",2.5468147e7,2.5468163e7,"""ACGAAGAGGGGGTGTTC""","""A""","""DNMT3A""","""p.E505fs*141""","""frameshift_variant""",0.0898,942.0,0,17,1,0,0,1,0,0,0,0,1,0,0,0,1,0,0,604,93,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""P131472""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",0.376635,355.550333,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null
"""P131505""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",0.376635,355.550333,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null
"""P131816""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",0.376635,355.550333,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null


In [115]:
mol_df["CHR"].value_counts().to_numpy()


array([['11', 326],
       ['22', 55],
       [None, 114],
       ['18', 128],
       ['12', 596],
       ['1', 365],
       ['3', 115],
       ['10', 18],
       ['6', 3],
       ['8', 41],
       ['5', 369],
       ['16', 91],
       ['13', 42],
       ['X', 1204],
       ['21', 866],
       ['7', 537],
       ['15', 192],
       ['14', 4],
       ['20', 999],
       ['2', 1513],
       ['19', 155],
       ['9', 116],
       ['17', 1387],
       ['4', 1699]], dtype=object)

In [116]:
def to_int(str):
    if(str == 'X'):
        return 23 # A justifier ?
    else:
        return int(str)

mol_df = mol_df.with_columns(
    pl.col("CHR").map_elements(to_int , return_dtype=pl.Int64)
    .alias("CHR")
)



# ORDINAL ENCODING

In [117]:
def dna_to_array(dna):
    lst = []
    
    nitrogen_bases = ['a','g','t','c']
    dna = dna.lower()
    
    
    for ch in dna:
        if(ch in nitrogen_bases):
            lst.append(ch)
        else:
            lst.append('n')
        
        
    return lst

def ordinal_encoder_dna(dna):
    mapping = {'a' : 0.25 , 'c' : 0.5 ,'g':0.75 , 't' : 1.00}
    
    nitrogen_bases = ['a','g','t','c']
    
    return [mapping[x] if x in nitrogen_bases else 0.00 for x in dna ]

    


seq_test = 'TTCAGCCAGTG'
ordinal_encoder_dna(dna_to_array(seq_test))





[1.0, 1.0, 0.5, 0.25, 0.75, 0.5, 0.5, 0.25, 0.75, 1.0, 0.75]

# K-MER ENCODING

In [118]:
def extract_kmers(sequence, k):
    return [sequence[i:i+k] for i in range(len(sequence) - k + 1)]


def join_str(str):
    return ' '.join(str)



k = 7 # K-mer
mySeq= 'TCTCACACATGTGCCAATCACTGTCACCC' # DNA Sequence


print(f"Example of transformation of DNA sequence : {mySeq} with {k}-mer")
new_str = join_str(extract_kmers(mySeq, k=7))

print(f"Transformation : {new_str} ")

Example of transformation of DNA sequence : TCTCACACATGTGCCAATCACTGTCACCC with 7-mer
Transformation : TCTCACA CTCACAC TCACACA CACACAT ACACATG CACATGT ACATGTG CATGTGC ATGTGCC TGTGCCA GTGCCAA TGCCAAT GCCAATC CCAATCA CAATCAC AATCACT ATCACTG TCACTGT CACTGTC ACTGTCA CTGTCAC TGTCACC GTCACCC 


In [119]:
from sklearn.feature_extraction.text import CountVectorizer


mySeq= 'TCTCACACATGTGCCAATCACTGTCACCC'
mySeq2 = 'GTGCCCAGGTTCAGTGAGTGACACAGGCAG'


DNA_list = [join_str(extract_kmers(mySeq,k=6)) , join_str(extract_kmers(mySeq2,k=6))]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(DNA_list).toarray()

print("Vocabulary: ", vectorizer.vocabulary_)


print(f"Encoded DNA : {X}")




Vocabulary:  {'tctcac': 41, 'ctcaca': 24, 'tcacac': 37, 'cacaca': 13, 'acacat': 2, 'cacatg': 15, 'acatgt': 4, 'catgtg': 20, 'atgtgc': 11, 'tgtgcc': 47, 'gtgcca': 34, 'tgccaa': 44, 'gccaat': 28, 'ccaatc': 21, 'caatca': 12, 'aatcac': 0, 'atcact': 10, 'tcactg': 39, 'cactgt': 16, 'actgtc': 5, 'ctgtca': 25, 'tgtcac': 46, 'gtcacc': 31, 'tcaccc': 38, 'gtgccc': 35, 'tgccca': 45, 'gcccag': 29, 'cccagg': 23, 'ccaggt': 22, 'caggtt': 18, 'aggttc': 7, 'ggttca': 30, 'gttcag': 36, 'ttcagt': 48, 'tcagtg': 40, 'cagtga': 19, 'agtgag': 9, 'gtgagt': 33, 'tgagtg': 43, 'gagtga': 27, 'agtgac': 8, 'gtgaca': 32, 'tgacac': 42, 'gacaca': 26, 'acacag': 1, 'cacagg': 14, 'acaggc': 3, 'caggca': 17, 'aggcag': 6}
Encoded DNA : [[1 0 1 0 1 1 0 0 0 0 1 1 1 1 0 1 1 0 0 0 1 1 0 0 1 1 0 0 1 0 0 1 0 0 1 0
  0 1 1 1 0 1 0 0 1 0 1 1 0]
 [0 1 0 1 0 0 1 1 1 1 0 0 0 0 1 0 0 1 1 1 0 0 1 1 0 0 1 1 0 1 1 0 1 1 0 1
  1 0 0 0 1 0 1 1 0 1 0 0 1]]


In [120]:
# TREAT THE REF with length <= 4 (arbitrary ?) and >4 differently ! 


k_mer_coef  = 4

mol_df = mol_df.with_columns(
    pl.when(pl.col("REF").str.len_chars() <= k_mer_coef)
    .then(pl.col("REF"))
    .otherwise(None) 
    .alias("REF_not_mer")
)




In [121]:
mol_df

ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH,is_complex_nucleotide,REF_length,ALT_length,is_transition,is_transversion,is_indel,REF_A,REF_C,REF_G,REF_T,ALT_A,ALT_C,ALT_G,ALT_T,complex_nucleotide_REF,complex_nucleotide_ALT,REF is_SNV,mutations_per_gene,mutations_per_gene_effect,mutations_per_region,REF_not_mer
str,i64,f64,f64,str,str,str,str,str,f64,f64,i16,u32,u32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,u32,u32,u32,str
"""P100000""",11,1.19149248e8,1.19149248e8,"""G""","""A""","""CBL""","""p.C419Y""","""non_synonymous_codon""",0.083,1308.0,0,1,1,1,0,0,0,0,1,0,1,0,0,0,0,0,1,228,163,1,"""G"""
"""P100000""",5,1.31822301e8,1.31822301e8,"""G""","""T""","""IRF1""","""p.Y164*""","""stop_gained""",0.022,532.0,0,1,1,0,1,0,0,0,1,0,0,0,0,1,0,0,1,22,4,1,"""G"""
"""P100000""",3,7.769406e7,7.769406e7,"""G""","""C""","""ROBO2""","""p.?""","""splice_site_variant""",0.41,876.0,0,1,1,0,1,0,0,0,1,0,0,1,0,0,0,0,1,17,3,3,"""G"""
"""P100000""",4,1.06164917e8,1.06164917e8,"""G""","""T""","""TET2""","""p.R1262L""","""non_synonymous_codon""",0.43,826.0,0,1,1,0,1,0,0,0,1,0,0,0,0,1,0,0,1,1663,417,4,"""G"""
"""P100000""",2,2.5468147e7,2.5468163e7,"""ACGAAGAGGGGGTGTTC""","""A""","""DNMT3A""","""p.E505fs*141""","""frameshift_variant""",0.0898,942.0,0,17,1,0,0,1,0,0,0,0,1,0,0,0,1,0,0,604,93,1,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""P131472""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",0.376635,355.550333,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null,null
"""P131505""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",0.376635,355.550333,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null,null
"""P131816""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",0.376635,355.550333,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null,null


In [122]:
import category_encoders as ce

mol_df_pd = mol_df.to_pandas()

encoder = ce.BinaryEncoder(cols=["REF_not_mer"])
mol_df = encoder.fit_transform(mol_df_pd)



mol_df = pl.from_pandas(mol_df)

mol_df



ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH,is_complex_nucleotide,REF_length,ALT_length,is_transition,is_transversion,is_indel,REF_A,REF_C,REF_G,REF_T,ALT_A,ALT_C,ALT_G,ALT_T,complex_nucleotide_REF,complex_nucleotide_ALT,REF is_SNV,mutations_per_gene,mutations_per_gene_effect,mutations_per_region,REF_not_mer_0,REF_not_mer_1,REF_not_mer_2,REF_not_mer_3,REF_not_mer_4,REF_not_mer_5,REF_not_mer_6
str,f64,f64,f64,str,str,str,str,str,f64,f64,f64,f64,f64,f64,i32,f64,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,u32,u32,f64,i64,i64,i64,i64,i64,i64,i64
"""P100000""",11.0,1.19149248e8,1.19149248e8,"""G""","""A""","""CBL""","""p.C419Y""","""non_synonymous_codon""",0.083,1308.0,0.0,1.0,1.0,1.0,0,0.0,0,0,1,0,1,0,0,0,0,0,1,228,163,1.0,0,0,0,0,0,0,1
"""P100000""",5.0,1.31822301e8,1.31822301e8,"""G""","""T""","""IRF1""","""p.Y164*""","""stop_gained""",0.022,532.0,0.0,1.0,1.0,0.0,1,0.0,0,0,1,0,0,0,0,1,0,0,1,22,4,1.0,0,0,0,0,0,0,1
"""P100000""",3.0,7.769406e7,7.769406e7,"""G""","""C""","""ROBO2""","""p.?""","""splice_site_variant""",0.41,876.0,0.0,1.0,1.0,0.0,1,0.0,0,0,1,0,0,1,0,0,0,0,1,17,3,3.0,0,0,0,0,0,0,1
"""P100000""",4.0,1.06164917e8,1.06164917e8,"""G""","""T""","""TET2""","""p.R1262L""","""non_synonymous_codon""",0.43,826.0,0.0,1.0,1.0,0.0,1,0.0,0,0,1,0,0,0,0,1,0,0,1,1663,417,4.0,0,0,0,0,0,0,1
"""P100000""",2.0,2.5468147e7,2.5468163e7,"""ACGAAGAGGGGGTGTTC""","""A""","""DNMT3A""","""p.E505fs*141""","""frameshift_variant""",0.0898,942.0,0.0,17.0,1.0,0.0,0,1.0,0,0,0,0,1,0,0,0,1,0,0,604,93,1.0,1,1,0,0,0,0,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""P131472""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",0.376635,355.550333,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null,1,1,0,0,0,0,1
"""P131505""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",0.376635,355.550333,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null,1,1,0,0,0,0,1
"""P131816""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",0.376635,355.550333,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null,1,1,0,0,0,0,1


# GENE


In [123]:
mol_df["GENE"].unique()

GENE
str
"""SETD2"""
"""ARID2"""
"""KIT"""
"""STAT3"""
"""ETV6"""
…
"""RUNX1"""
"""SMG1"""
"""ZRSR2"""


# ANCIENNE NOMENCLATURE => NOUVELLES NOMENCLATURES

In [124]:
# MLL => KMT2A
# WHSC1 => NSD2
# H3F3A => H3-3A 
# FAM175A => ABRAXAS1
# PAPD5 => TENT4B

# NATIONAL LIBRARY OF MEDICINE

mapping = {'MLL' : 'KMT2A' , 'WHSC1' : 'NSD2' , 'H3F3A' : 'H3-3A' , 'FAM175A' : 'ABRAXAS1' , 'PAPD5' : 'TENT4B'}

mol_df = mol_df.with_columns(
    pl.col("GENE").map_elements(lambda g: mapping.get(g, g)).alias("GENE")
)

# EXPLIQUER CETTE ETAPE

C:\Users\zakar\AppData\Local\Temp\ipykernel_16716\126878532.py:11: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  mol_df = mol_df.with_columns(


In [125]:
mol_df

ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH,is_complex_nucleotide,REF_length,ALT_length,is_transition,is_transversion,is_indel,REF_A,REF_C,REF_G,REF_T,ALT_A,ALT_C,ALT_G,ALT_T,complex_nucleotide_REF,complex_nucleotide_ALT,REF is_SNV,mutations_per_gene,mutations_per_gene_effect,mutations_per_region,REF_not_mer_0,REF_not_mer_1,REF_not_mer_2,REF_not_mer_3,REF_not_mer_4,REF_not_mer_5,REF_not_mer_6
str,f64,f64,f64,str,str,str,str,str,f64,f64,f64,f64,f64,f64,i32,f64,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,u32,u32,f64,i64,i64,i64,i64,i64,i64,i64
"""P100000""",11.0,1.19149248e8,1.19149248e8,"""G""","""A""","""CBL""","""p.C419Y""","""non_synonymous_codon""",0.083,1308.0,0.0,1.0,1.0,1.0,0,0.0,0,0,1,0,1,0,0,0,0,0,1,228,163,1.0,0,0,0,0,0,0,1
"""P100000""",5.0,1.31822301e8,1.31822301e8,"""G""","""T""","""IRF1""","""p.Y164*""","""stop_gained""",0.022,532.0,0.0,1.0,1.0,0.0,1,0.0,0,0,1,0,0,0,0,1,0,0,1,22,4,1.0,0,0,0,0,0,0,1
"""P100000""",3.0,7.769406e7,7.769406e7,"""G""","""C""","""ROBO2""","""p.?""","""splice_site_variant""",0.41,876.0,0.0,1.0,1.0,0.0,1,0.0,0,0,1,0,0,1,0,0,0,0,1,17,3,3.0,0,0,0,0,0,0,1
"""P100000""",4.0,1.06164917e8,1.06164917e8,"""G""","""T""","""TET2""","""p.R1262L""","""non_synonymous_codon""",0.43,826.0,0.0,1.0,1.0,0.0,1,0.0,0,0,1,0,0,0,0,1,0,0,1,1663,417,4.0,0,0,0,0,0,0,1
"""P100000""",2.0,2.5468147e7,2.5468163e7,"""ACGAAGAGGGGGTGTTC""","""A""","""DNMT3A""","""p.E505fs*141""","""frameshift_variant""",0.0898,942.0,0.0,17.0,1.0,0.0,0,1.0,0,0,0,0,1,0,0,0,1,0,0,604,93,1.0,1,1,0,0,0,0,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""P131472""",null,null,null,null,null,"""KMT2A""","""MLL_PTD""","""PTD""",0.376635,355.550333,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null,1,1,0,0,0,0,1
"""P131505""",null,null,null,null,null,"""KMT2A""","""MLL_PTD""","""PTD""",0.376635,355.550333,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null,1,1,0,0,0,0,1
"""P131816""",null,null,null,null,null,"""KMT2A""","""MLL_PTD""","""PTD""",0.376635,355.550333,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null,1,1,0,0,0,0,1


# CYTOGENETICS

In [126]:
import mygene

genes = list(mol_df["GENE"].unique())

mg = mygene.MyGeneInfo()

results = mg.querymany(genes, scopes="symbol", fields='go', species="human")

df_pathways = pd.DataFrame(results)

results

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed


[{'query': 'H3-3A',
  '_id': '3020',
  '_score': 20.942583,
  'go': {'BP': [{'evidence': 'IEA',
     'gocategory': 'BP',
     'id': 'GO:0001556',
     'qualifier': 'involved_in',
     'term': 'oocyte maturation'},
    {'evidence': 'IEA',
     'gocategory': 'BP',
     'id': 'GO:0001649',
     'qualifier': 'acts_upstream_of_or_within',
     'term': 'osteoblast differentiation'},
    {'evidence': 'IDA',
     'gocategory': 'BP',
     'id': 'GO:0006334',
     'pubmed': 21636898,
     'qualifier': 'involved_in',
     'term': 'nucleosome assembly'},
    {'evidence': 'IMP',
     'gocategory': 'BP',
     'id': 'GO:0006334',
     'pubmed': 25615412,
     'qualifier': 'involved_in',
     'term': 'nucleosome assembly'},
    {'evidence': 'IEA',
     'gocategory': 'BP',
     'id': 'GO:0006997',
     'qualifier': 'acts_upstream_of_or_within',
     'term': 'nucleus organization'},
    {'evidence': 'IEA',
     'gocategory': 'BP',
     'id': 'GO:0007283',
     'qualifier': 'acts_upstream_of_or_within',


In [127]:
gene_to_go = {}
for res in results:
    gene = res.get('query')
    go_bp = res.get('go', {}).get('BP', [])
    if isinstance(go_bp, dict):  # Cas d'un seul terme
        go_bp = [go_bp]
    go_terms = [go['id'] for go in go_bp if 'id' in go]
    if gene and go_terms:
        gene_to_go[gene] = go_terms
        

gene_to_go

{'H3-3A': ['GO:0001556',
  'GO:0001649',
  'GO:0006334',
  'GO:0006334',
  'GO:0006997',
  'GO:0007283',
  'GO:0007286',
  'GO:0007338',
  'GO:0007566',
  'GO:0008283',
  'GO:0008584',
  'GO:0030307',
  'GO:0031508',
  'GO:0031509',
  'GO:0032200',
  'GO:0035264',
  'GO:0042692',
  'GO:0048477',
  'GO:0090230',
  'GO:1902340'],
 'JAK3': ['GO:0002250',
  'GO:0002376',
  'GO:0002731',
  'GO:0007167',
  'GO:0007259',
  'GO:0007259',
  'GO:0007260',
  'GO:0010561',
  'GO:0019221',
  'GO:0019221',
  'GO:0022407',
  'GO:0030154',
  'GO:0030183',
  'GO:0032693',
  'GO:0032695',
  'GO:0035556',
  'GO:0035556',
  'GO:0035556',
  'GO:0035723',
  'GO:0035771',
  'GO:0035771',
  'GO:0038110',
  'GO:0038110',
  'GO:0038111',
  'GO:0038113',
  'GO:0042981',
  'GO:0043029',
  'GO:0043029',
  'GO:0045087',
  'GO:0045626',
  'GO:0046425',
  'GO:0050865',
  'GO:0050868',
  'GO:0060397',
  'GO:0060397',
  'GO:0070232',
  'GO:0070244',
  'GO:0070669',
  'GO:0070670',
  'GO:0070672',
  'GO:0071104',
  'GO:

In [128]:
from sklearn.preprocessing import MultiLabelBinarizer

# 4. Créer une matrice binaire gène × GO term
mlb = MultiLabelBinarizer()
go_matrix = mlb.fit_transform(gene_to_go.values())
go_df = pd.DataFrame(go_matrix, index=gene_to_go.keys(), columns=mlb.classes_)

# 5. Filtrer les GO terms peu fréquents (seuil = 5 gènes)
min_gene_count = 5
filtered_go_df = go_df.loc[:, (go_df.sum(axis=0) >= min_gene_count)]

# 6. Résultat final
print(f"Dimensions finales : {filtered_go_df.shape}")
filtered_go_df

Dimensions finales : (124, 100)


,GO:0000082,GO:0000122,GO:0000165,GO:0000398,GO:0001666,GO:0001701,GO:0001889,GO:0002376,GO:0006281,GO:0006302,...,GO:0060038,GO:0060253,GO:0070374,GO:0071222,GO:0071456,GO:0071466,GO:0090307,GO:0090398,GO:1902895,GO:2000045
H3-3A,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
JAK3,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
ASXL1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
IDH2,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
JARID2,0,1,0,0,0,0,1,0,0,0,...,1,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
HIPK2,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
SF1,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
DICER1,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
KIT,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [129]:
mol_df_pd = mol_df.to_pandas()

df_merged = mol_df_pd.merge(filtered_go_df, left_on='GENE', right_index=True, how='left')



In [130]:
df_merged = pl.from_pandas(df_merged)

In [131]:
df_merged

ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH,is_complex_nucleotide,REF_length,ALT_length,is_transition,is_transversion,is_indel,REF_A,REF_C,REF_G,REF_T,ALT_A,ALT_C,ALT_G,ALT_T,complex_nucleotide_REF,complex_nucleotide_ALT,REF is_SNV,mutations_per_gene,mutations_per_gene_effect,mutations_per_region,REF_not_mer_0,REF_not_mer_1,REF_not_mer_2,REF_not_mer_3,REF_not_mer_4,REF_not_mer_5,…,GO:0035556,GO:0035914,GO:0036211,GO:0042127,GO:0042981,GO:0043065,GO:0043066,GO:0043410,GO:0043491,GO:0043524,GO:0045087,GO:0045747,GO:0045766,GO:0045892,GO:0045893,GO:0045944,GO:0046777,GO:0048511,GO:0048709,GO:0048873,GO:0050821,GO:0051276,GO:0051301,GO:0051321,GO:0051402,GO:0051726,GO:0051897,GO:0060038,GO:0060253,GO:0070374,GO:0071222,GO:0071456,GO:0071466,GO:0090307,GO:0090398,GO:1902895,GO:2000045
str,f64,f64,f64,str,str,str,str,str,f64,f64,f64,f64,f64,f64,i32,f64,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,u32,u32,f64,i64,i64,i64,i64,i64,i64,…,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""P100000""",11.0,1.19149248e8,1.19149248e8,"""G""","""A""","""CBL""","""p.C419Y""","""non_synonymous_codon""",0.083,1308.0,0.0,1.0,1.0,1.0,0,0.0,0,0,1,0,1,0,0,0,0,0,1,228,163,1.0,0,0,0,0,0,0,…,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0
"""P100000""",5.0,1.31822301e8,1.31822301e8,"""G""","""T""","""IRF1""","""p.Y164*""","""stop_gained""",0.022,532.0,0.0,1.0,1.0,0.0,1,0.0,0,0,1,0,0,0,0,1,0,0,1,22,4,1.0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,1,0,0,1,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
"""P100000""",3.0,7.769406e7,7.769406e7,"""G""","""C""","""ROBO2""","""p.?""","""splice_site_variant""",0.41,876.0,0.0,1.0,1.0,0.0,1,0.0,0,0,1,0,0,1,0,0,0,0,1,17,3,3.0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""P100000""",4.0,1.06164917e8,1.06164917e8,"""G""","""T""","""TET2""","""p.R1262L""","""non_synonymous_codon""",0.43,826.0,0.0,1.0,1.0,0.0,1,0.0,0,0,1,0,0,0,0,1,0,0,1,1663,417,4.0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""P100000""",2.0,2.5468147e7,2.5468163e7,"""ACGAAGAGGGGGTGTTC""","""A""","""DNMT3A""","""p.E505fs*141""","""frameshift_variant""",0.0898,942.0,0.0,17.0,1.0,0.0,0,1.0,0,0,0,0,1,0,0,0,1,0,0,604,93,1.0,1,1,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""P131472""",null,null,null,null,null,"""KMT2A""","""MLL_PTD""","""PTD""",0.376635,355.550333,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null,1,1,0,0,0,0,…,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""P131505""",null,null,null,null,null,"""KMT2A""","""MLL_PTD""","""PTD""",0.376635,355.550333,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null,1,1,0,0,0,0,…,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""P131816""",null,null,null,null,null,"""KMT2A""","""MLL_PTD""","""PTD""",0.376635,355.550333,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null,1,1,0,0,0,0,…,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [132]:
df_merged.filter(pl.col("GENE") == "FLT3")

ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH,is_complex_nucleotide,REF_length,ALT_length,is_transition,is_transversion,is_indel,REF_A,REF_C,REF_G,REF_T,ALT_A,ALT_C,ALT_G,ALT_T,complex_nucleotide_REF,complex_nucleotide_ALT,REF is_SNV,mutations_per_gene,mutations_per_gene_effect,mutations_per_region,REF_not_mer_0,REF_not_mer_1,REF_not_mer_2,REF_not_mer_3,REF_not_mer_4,REF_not_mer_5,…,GO:0035556,GO:0035914,GO:0036211,GO:0042127,GO:0042981,GO:0043065,GO:0043066,GO:0043410,GO:0043491,GO:0043524,GO:0045087,GO:0045747,GO:0045766,GO:0045892,GO:0045893,GO:0045944,GO:0046777,GO:0048511,GO:0048709,GO:0048873,GO:0050821,GO:0051276,GO:0051301,GO:0051321,GO:0051402,GO:0051726,GO:0051897,GO:0060038,GO:0060253,GO:0070374,GO:0071222,GO:0071456,GO:0071466,GO:0090307,GO:0090398,GO:1902895,GO:2000045
str,f64,f64,f64,str,str,str,str,str,f64,f64,f64,f64,f64,f64,i32,f64,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,u32,u32,f64,i64,i64,i64,i64,i64,i64,…,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""P102824""",13.0,2.8592641e7,2.8592641e7,"""T""","""A""","""FLT3""","""p.D835V""","""non_synonymous_codon""",0.178,704.0,0.0,1.0,1.0,0.0,1,0.0,0,0,0,1,1,0,0,0,0,0,1,56,30,3.0,0,0,0,0,1,0,…,0,0,0,1,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
"""P102954""",13.0,2.8608107e7,2.8608107e7,"""G""","""A""","""FLT3""","""p.A620V""","""non_synonymous_codon""",0.09,2050.0,0.0,1.0,1.0,1.0,0,0.0,0,0,1,0,1,0,0,0,0,0,1,56,30,1.0,0,0,0,0,0,0,…,0,0,0,1,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
"""P103084""",13.0,2.860242e7,2.860242e7,"""C""","""T""","""FLT3""","""p.A650T""","""non_synonymous_codon""",0.48,1124.0,0.0,1.0,1.0,0.0,0,0.0,0,1,0,0,0,0,0,1,0,0,1,56,30,1.0,0,0,0,0,0,1,…,0,0,0,1,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
"""P105773""",13.0,2.8592642e7,2.8592642e7,"""C""","""G""","""FLT3""","""p.D835H""","""non_synonymous_codon""",0.245,2372.0,0.0,1.0,1.0,0.0,1,0.0,0,1,0,0,0,0,1,0,0,0,1,56,30,7.0,0,0,0,0,0,1,…,0,0,0,1,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
"""P105775""",13.0,2.8592642e7,2.8592642e7,"""C""","""T""","""FLT3""","""p.D835N""","""non_synonymous_codon""",0.065,2081.0,0.0,1.0,1.0,0.0,0,0.0,0,1,0,0,0,0,0,1,0,0,1,56,30,7.0,0,0,0,0,0,1,…,0,0,0,1,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""P118243""",null,null,null,null,null,"""FLT3""","""FLT3_ITD""","""ITD""",0.333821,989.35307,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,56,26,null,1,1,0,0,0,0,…,0,0,0,1,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
"""P120352""",null,null,null,null,null,"""FLT3""","""FLT3_ITD""","""ITD""",0.131665,1034.545778,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,56,26,null,1,1,0,0,0,0,…,0,0,0,1,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
"""P116526""",null,null,null,null,null,"""FLT3""","""FLT3_ITD""","""ITD""",0.283986,1130.100714,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,56,26,null,1,1,0,0,0,0,…,0,0,0,1,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0


In [133]:
# not_found_gene = ['MLL', 'WHSC1', 'H3F3A', 'PAPD5', 'FAM175A']

# df_merged = df_merged.with_columns(
#     pl.when(pl.col("GENE").is_in(not_found_gene))
#     .then(pl.col("GENE"))
#     .otherwise(None)
#     .alias("not_found_gene")
# )

# PARLER DE CA



In [134]:
# df_merged = df_merged.with_columns(
#     pl.when(pl.col("GENE").is_in(not_found_gene))
#     .then(0)
#     .otherwise(1)
#     .alias("Found_in_my_Gene")
# )

In [135]:
mol_df = df_merged

# NULL VALUES : CHR , START , END

In [136]:
quant_vars = ["CHR" , "START" , "END"]
sub_df = mol_df.select(quant_vars)

# Convert to pandas
sub_pd = sub_df.to_pandas()
 
# Apply Model-Based Imputation (Random Forest)
imputer = IterativeImputer(estimator=RandomForestRegressor(), random_state=42)
imputed_data = imputer.fit_transform(sub_pd)

# 4. To Polars
imputed_pl = pl.DataFrame(imputed_data, schema=sub_df.columns)

# 5. Replace nulls
mol_df = mol_df.with_columns([imputed_pl[col].alias(col) for col in quant_vars])

In [137]:
mol_df

ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH,is_complex_nucleotide,REF_length,ALT_length,is_transition,is_transversion,is_indel,REF_A,REF_C,REF_G,REF_T,ALT_A,ALT_C,ALT_G,ALT_T,complex_nucleotide_REF,complex_nucleotide_ALT,REF is_SNV,mutations_per_gene,mutations_per_gene_effect,mutations_per_region,REF_not_mer_0,REF_not_mer_1,REF_not_mer_2,REF_not_mer_3,REF_not_mer_4,REF_not_mer_5,…,GO:0035556,GO:0035914,GO:0036211,GO:0042127,GO:0042981,GO:0043065,GO:0043066,GO:0043410,GO:0043491,GO:0043524,GO:0045087,GO:0045747,GO:0045766,GO:0045892,GO:0045893,GO:0045944,GO:0046777,GO:0048511,GO:0048709,GO:0048873,GO:0050821,GO:0051276,GO:0051301,GO:0051321,GO:0051402,GO:0051726,GO:0051897,GO:0060038,GO:0060253,GO:0070374,GO:0071222,GO:0071456,GO:0071466,GO:0090307,GO:0090398,GO:1902895,GO:2000045
str,f64,f64,f64,str,str,str,str,str,f64,f64,f64,f64,f64,f64,i32,f64,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,u32,u32,f64,i64,i64,i64,i64,i64,i64,…,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""P100000""",11.0,1.19149248e8,1.19149248e8,"""G""","""A""","""CBL""","""p.C419Y""","""non_synonymous_codon""",0.083,1308.0,0.0,1.0,1.0,1.0,0,0.0,0,0,1,0,1,0,0,0,0,0,1,228,163,1.0,0,0,0,0,0,0,…,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0
"""P100000""",5.0,1.31822301e8,1.31822301e8,"""G""","""T""","""IRF1""","""p.Y164*""","""stop_gained""",0.022,532.0,0.0,1.0,1.0,0.0,1,0.0,0,0,1,0,0,0,0,1,0,0,1,22,4,1.0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,1,0,0,1,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
"""P100000""",3.0,7.769406e7,7.769406e7,"""G""","""C""","""ROBO2""","""p.?""","""splice_site_variant""",0.41,876.0,0.0,1.0,1.0,0.0,1,0.0,0,0,1,0,0,1,0,0,0,0,1,17,3,3.0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""P100000""",4.0,1.06164917e8,1.06164917e8,"""G""","""T""","""TET2""","""p.R1262L""","""non_synonymous_codon""",0.43,826.0,0.0,1.0,1.0,0.0,1,0.0,0,0,1,0,0,0,0,1,0,0,1,1663,417,4.0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""P100000""",2.0,2.5468147e7,2.5468163e7,"""ACGAAGAGGGGGTGTTC""","""A""","""DNMT3A""","""p.E505fs*141""","""frameshift_variant""",0.0898,942.0,0.0,17.0,1.0,0.0,0,1.0,0,0,0,0,1,0,0,0,1,0,0,604,93,1.0,1,1,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""P131472""",3.0,7.9008e7,7.9009e7,null,null,"""KMT2A""","""MLL_PTD""","""PTD""",0.376635,355.550333,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null,1,1,0,0,0,0,…,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""P131505""",3.0,7.9008e7,7.9009e7,null,null,"""KMT2A""","""MLL_PTD""","""PTD""",0.376635,355.550333,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null,1,1,0,0,0,0,…,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""P131816""",3.0,7.9008e7,7.9009e7,null,null,"""KMT2A""","""MLL_PTD""","""PTD""",0.376635,355.550333,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null,1,1,0,0,0,0,…,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


# PROTEIN_CHANGE

In [138]:

# ON GERE LES PROTEIN CHANGE QUI COMMENCE PAR p.

import re 


protein_change = list(mol_df["PROTEIN_CHANGE"].unique())


protein_change

uppercase_alphabet = [chr(i) for i in range(ord('A'), ord('Z') + 1)]

aa_fullname = {
    'A': 'Alanine',
    'C': 'Cystéine',
    'D': 'Acide aspartique',
    'E': 'Acide glutamique',
    'F': 'Phénylalanine',
    'G': 'Glycine',
    'H': 'Histidine',
    'I': 'Isoleucine',
    'K': 'Lysine',
    'L': 'Leucine',
    'M': 'Méthionine',
    'N': 'Asparagine',
    'P': 'Proline',
    'Q': 'Glutamine',
    'R': 'Arginine',
    'S': 'Sérine',
    'T': 'Thréonine',
    'V': 'Valine',
    'W': 'Tryptophane',
    'Y': 'Tyrosine'
}




def protein_name(protein_changes, ch):
    lst_prot = []
    pattern = rf"p\.{ch}\d+.*"

    for prot in protein_changes:
        if isinstance(prot, str): 
            prot_name = re.findall(pattern, prot)
            if prot_name:
                lst_prot+=prot_name
                
    return lst_prot
        
        
dico_protein_changes = {}
        
for ch in uppercase_alphabet:
    lst  = protein_name(protein_change , ch)
    if(len(lst) >=1):
        ch_name = aa_fullname.get(ch)
        dico_protein_changes[ch_name] = lst

print(dico_protein_changes)


# A , C , D , E , F , G , H , I , K , L , M , N , P , Q , R , S , T , V , W , Y


{'Alanine': ['p.A858fs*29', 'p.A36V', 'p.A435S', 'p.A640fs*14', 'p.A571fs*5', 'p.A138T', 'p.A59E', 'p.A671fs*29', 'p.A809fs*13', 'p.A1810fs*10', 'p.A146V', 'p.A377G', 'p.A1158V', 'p.A51fs*36', 'p.A324fs*276', 'p.A1196delA', 'p.A59H', 'p.A716fs*9', 'p.A871T', 'p.A59G', 'p.A286T', 'p.A68fs*70', 'p.A787V', 'p.A972V', 'p.A658*', 'p.A159P', 'p.A147fs*13', 'p.A338fs*262', 'p.A1341P', 'p.A620V', 'p.A1331T', 'p.A113fs*86', 'p.A1189fs*4', 'p.A1166fs*56', 'p.A1505T', 'p.A159V', 'p.A235fs*1', 'p.A72S', 'p.A1196fs*2', 'p.A142fs*4', 'p.A382fs*6', 'p.A380T', 'p.A1159_I1160delinsV', 'p.A377V', 'p.A187_T188delAT', 'p.A85T', 'p.A368fs*39', 'p.A696V', 'p.A488T', 'p.A385fs*57', 'p.A161D', 'p.A1618fs*56', 'p.A996fs*13', 'p.A1224E', 'p.A277fs*16', 'p.A952fs*6', 'p.A64fs*64', 'p.A972T', 'p.A258fs*3', 'p.A707fs*10', 'p.A1240V', 'p.A213V', 'p.A60V', 'p.A187T', 'p.A322fs*4', 'p.A187G', 'p.A520fs*4', 'p.A550fs*11', 'p.A903T', 'p.A1219fs*4', 'p.A382fs*69', 'p.A1512V', 'p.A1224delA', 'p.A40fs*120', 'p.A216G', 'p.

In [139]:

acide_aminee = {}

for elem in protein_change:
    for aa_name , mutations in dico_protein_changes.items():
        if elem in mutations:
            acide_aminee[elem] = aa_name
            

print(acide_aminee)

{'p.R1878fs*2': 'Arginine', 'p.P904Q': 'Proline', 'p.G1152E': 'Glycine', 'p.D2064fs*38': 'Acide aspartique', 'p.D233E': 'Acide aspartique', 'p.S210*': 'Sérine', 'p.G12D': 'Glycine', 'p.G244D': 'Glycine', 'p.S100*': 'Sérine', 'p.V41I': 'Valine', 'p.T590fs*29': 'Thréonine', 'p.S327fs*4': 'Sérine', 'p.S1255*': 'Sérine', 'p.S1598fs*28': 'Sérine', 'p.N460fs*5': 'Asparagine', 'p.E404K': 'Acide glutamique', 'p.Q1687*': 'Glutamine', 'p.Q317*': 'Glutamine', 'p.C135F': 'Cystéine', 'p.L286_F287insSLSL': 'Leucine', 'p.R583*': 'Arginine', 'p.Y220C': 'Tyrosine', 'p.Q124*': 'Glutamine', 'p.R1262L': 'Arginine', 'p.L98fs*24': 'Leucine', 'p.D664N': 'Acide aspartique', 'p.L1694*': 'Leucine', 'p.N253fs*41': 'Asparagine', 'p.G422fs*5': 'Glycine', 'p.W408C': 'Tryptophane', 'p.H302P': 'Histidine', 'p.V371fs*18': 'Valine', 'p.Y346C': 'Tyrosine', 'p.L161P': 'Leucine', 'p.Q432*': 'Glutamine', 'p.V360fs*234': 'Valine', 'p.R174fs*4': 'Arginine', 'p.I1873S': 'Isoleucine', 'p.K1173*': 'Lysine', 'p.R267W': 'Arginine

In [140]:
# KMT2A => MLL PTD => Lysine 
# FLT3-ID => FLT3 => tyrosine

acide_aminee["MLL_PTD"] = "Lysine"
acide_aminee["FLT3_ITD"] = "Tyrosine"
acide_aminee["p.?"] = "Unknown"

In [141]:
mol_df = mol_df.with_columns(
    pl.col("PROTEIN_CHANGE").map_elements(lambda x: acide_aminee.get(x, None)).alias("AA_TYPE")
)

C:\Users\zakar\AppData\Local\Temp\ipykernel_16716\1629844449.py:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  mol_df = mol_df.with_columns(


In [142]:
mol_df["AA_TYPE"].value_counts()

AA_TYPE,count
str,u32
null,16
"""Tryptophane""",255
"""Lysine""",904
"""Glutamine""",783
"""Acide aspartique""",339
…,…
"""Valine""",318
"""Phénylalanine""",165
"""Thréonine""",267


In [143]:
mol_df.filter(pl.col("AA_TYPE").is_null())['PROTEIN_CHANGE'].value_counts()

PROTEIN_CHANGE,count
str,u32
"""p.*342S""",1
null,12
"""p.*636W""",1
"""p.*636C""",1
"""p.*342*""",1


In [144]:

protein_remove = ["p.*636C"	 , "p.*342*" , "p.*342S" , "p.*636W" ]

for elem in protein_remove:
    mol_df = mol_df.remove(pl.col("PROTEIN_CHANGE") == elem)

In [145]:
mol_df.filter(pl.col("AA_TYPE").is_null())['PROTEIN_CHANGE'].value_counts()

PROTEIN_CHANGE,count
str,u32
null,12


In [146]:
mol_df = mol_df.drop_nulls(subset="PROTEIN_CHANGE")

In [147]:
mol_df.filter(pl.col("AA_TYPE").is_null())['PROTEIN_CHANGE'].value_counts()

PROTEIN_CHANGE,count
str,u32


In [148]:
mol_df = mol_df.with_columns(
    pl.when(pl.col("PROTEIN_CHANGE") == "p.?")
    .then(1)
    .otherwise(0)
    .alias("IS_PT_UKNOWN")
)

In [149]:
import category_encoders as ce

mol_df_pd = mol_df.to_pandas()

encoder = ce.BinaryEncoder(cols=["AA_TYPE"])
mol_df = encoder.fit_transform(mol_df_pd)



mol_df = pl.from_pandas(mol_df)

mol_df



ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH,is_complex_nucleotide,REF_length,ALT_length,is_transition,is_transversion,is_indel,REF_A,REF_C,REF_G,REF_T,ALT_A,ALT_C,ALT_G,ALT_T,complex_nucleotide_REF,complex_nucleotide_ALT,REF is_SNV,mutations_per_gene,mutations_per_gene_effect,mutations_per_region,REF_not_mer_0,REF_not_mer_1,REF_not_mer_2,REF_not_mer_3,REF_not_mer_4,REF_not_mer_5,…,GO:0043066,GO:0043410,GO:0043491,GO:0043524,GO:0045087,GO:0045747,GO:0045766,GO:0045892,GO:0045893,GO:0045944,GO:0046777,GO:0048511,GO:0048709,GO:0048873,GO:0050821,GO:0051276,GO:0051301,GO:0051321,GO:0051402,GO:0051726,GO:0051897,GO:0060038,GO:0060253,GO:0070374,GO:0071222,GO:0071456,GO:0071466,GO:0090307,GO:0090398,GO:1902895,GO:2000045,AA_TYPE_0,AA_TYPE_1,AA_TYPE_2,AA_TYPE_3,AA_TYPE_4,IS_PT_UKNOWN
str,f64,f64,f64,str,str,str,str,str,f64,f64,f64,f64,f64,f64,i32,f64,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,u32,u32,f64,i64,i64,i64,i64,i64,i64,…,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i32
"""P100000""",11.0,1.19149248e8,1.19149248e8,"""G""","""A""","""CBL""","""p.C419Y""","""non_synonymous_codon""",0.083,1308.0,0.0,1.0,1.0,1.0,0,0.0,0,0,1,0,1,0,0,0,0,0,1,228,163,1.0,0,0,0,0,0,0,…,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0
"""P100000""",5.0,1.31822301e8,1.31822301e8,"""G""","""T""","""IRF1""","""p.Y164*""","""stop_gained""",0.022,532.0,0.0,1.0,1.0,0.0,1,0.0,0,0,1,0,0,0,0,1,0,0,1,22,4,1.0,0,0,0,0,0,0,…,0,0,0,0,1,0,0,1,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
"""P100000""",3.0,7.769406e7,7.769406e7,"""G""","""C""","""ROBO2""","""p.?""","""splice_site_variant""",0.41,876.0,0.0,1.0,1.0,0.0,1,0.0,0,0,1,0,0,1,0,0,0,0,1,17,3,3.0,0,0,0,0,0,0,…,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1
"""P100000""",4.0,1.06164917e8,1.06164917e8,"""G""","""T""","""TET2""","""p.R1262L""","""non_synonymous_codon""",0.43,826.0,0.0,1.0,1.0,0.0,1,0.0,0,0,1,0,0,0,0,1,0,0,1,1663,417,4.0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
"""P100000""",2.0,2.5468147e7,2.5468163e7,"""ACGAAGAGGGGGTGTTC""","""A""","""DNMT3A""","""p.E505fs*141""","""frameshift_variant""",0.0898,942.0,0.0,17.0,1.0,0.0,0,1.0,0,0,0,0,1,0,0,0,1,0,0,604,93,1.0,1,1,0,0,0,0,…,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""P131472""",3.0,7.9008e7,7.9009e7,null,null,"""KMT2A""","""MLL_PTD""","""PTD""",0.376635,355.550333,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null,1,1,0,0,0,0,…,0,0,0,0,0,0,0,0,1,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0
"""P131505""",3.0,7.9008e7,7.9009e7,null,null,"""KMT2A""","""MLL_PTD""","""PTD""",0.376635,355.550333,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null,1,1,0,0,0,0,…,0,0,0,0,0,0,0,0,1,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0
"""P131816""",3.0,7.9008e7,7.9009e7,null,null,"""KMT2A""","""MLL_PTD""","""PTD""",0.376635,355.550333,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null,1,1,0,0,0,0,…,0,0,0,0,0,0,0,0,1,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0


In [150]:
# Frameshift : a sequence change between the translation initiation (start) and termination (stop) codon where, compared to a reference sequence, 
# translation shifts to another reading frame.


# FrameShift protein_change
for elem in protein_change:
    if elem is not None and 'fs' in elem:
        print(elem)
        
mol_df = mol_df.with_columns(
    pl.when(pl.col("PROTEIN_CHANGE").str.contains('fs'))
    .then(1)
    .otherwise(0)
    .alias("is_frameshift")
)

p.R1878fs*2
p.D2064fs*38
p.T590fs*29
p.S327fs*4
p.S1598fs*28
p.N460fs*5
p.L98fs*24
p.N253fs*41
p.G422fs*5
p.V371fs*18
p.V360fs*234
p.R174fs*4
p.S206fs*1
p.Y2285fs*1
p.V509fs*14
p.A858fs*29
p.L546fs*1
p.M725fs*16
p.R370fs*7
p.I451fs*2
p.R169fs*56
p.P1496fs*75
p.P489fs*40
p.R198fs*8
p.V116fs*28
p.E143fs*2
p.K291fs*18
p.F468fs*5
p.C407fs*2
p.E227fs*1
p.R450fs*20
p.Y355fs*1
p.S1431fs*17
p.P442fs*16
p.E513fs*16
p.K228fs*25
p.D1412fs*12
p.Q1117fs*18
p.L85fs*28
p.E343fs*2
p.G170fs*7
p.L1418fs*7
p.S121fs*2
p.T1085fs*19
p.D596fs*1
p.E1128fs*7
p.R15fs*82
p.F158fs*18
p.D563fs*13
p.L1413fs*83
p.E476fs*7
p.N492fs*9
p.Q769fs*44
p.R380fs*72
p.G870fs*38
p.S1059fs*1
p.K618fs*1
p.V238fs*16
p.D123fs*11
p.Q289fs*11
p.N405fs*21
p.Q269fs*12
p.F183fs*23
p.S825fs*1
p.P250fs*3
p.S336fs*45
p.D1635fs*5
p.L743fs*57
p.N1573fs*6
p.H839fs*2
p.F1104fs*2
p.Q2230fs*114
p.N1504fs*67
p.S597fs*41
p.R103fs*9
p.A640fs*14
p.I254fs*32
p.L335fs*69
p.I1897fs*11
p.D1152fs*14
p.S1982fs*30
p.E773fs*12
p.V360fs*239
p.E294fs*22
p.E1

In [151]:
#NON_SENS_MUTATION


mol_df = mol_df.with_columns(
    pl.when(pl.col("PROTEIN_CHANGE").str.contains("\*"))
    .then(1)
    .otherwise(0)
    .alias("is_non_sens_mutation")
)

<>:5: SyntaxWarning: invalid escape sequence '\*'
<>:5: SyntaxWarning: invalid escape sequence '\*'
C:\Users\zakar\AppData\Local\Temp\ipykernel_16716\402174943.py:5: SyntaxWarning: invalid escape sequence '\*'
  pl.when(pl.col("PROTEIN_CHANGE").str.contains("\*"))


In [152]:
# IS MISSENSE

def ismissense(elem):
    if elem is not None and isinstance(elem, str):
        # Vérifie que le dernier caractère est une lettre majuscule (acide aminé)
        if elem.startswith("p.") and elem[-1].isalpha() and "*" not in elem and "fs" not in elem:
            return 1
    return 0

mol_df = mol_df.with_columns(
    pl.col("PROTEIN_CHANGE").map_elements(ismissense, return_dtype=pl.Int64).alias("IS_MISSENSE")
)


In [153]:
for elem in protein_change:
    if (elem is not None and "=" in elem):
        print(elem)

In [154]:
mol_df.filter(pl.col("EFFECT") == "PTD")



# j'ai déja traité : frameshift , stopgained , non_synonoumous_codon , splice_site_variant 

ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH,is_complex_nucleotide,REF_length,ALT_length,is_transition,is_transversion,is_indel,REF_A,REF_C,REF_G,REF_T,ALT_A,ALT_C,ALT_G,ALT_T,complex_nucleotide_REF,complex_nucleotide_ALT,REF is_SNV,mutations_per_gene,mutations_per_gene_effect,mutations_per_region,REF_not_mer_0,REF_not_mer_1,REF_not_mer_2,REF_not_mer_3,REF_not_mer_4,REF_not_mer_5,…,GO:0043524,GO:0045087,GO:0045747,GO:0045766,GO:0045892,GO:0045893,GO:0045944,GO:0046777,GO:0048511,GO:0048709,GO:0048873,GO:0050821,GO:0051276,GO:0051301,GO:0051321,GO:0051402,GO:0051726,GO:0051897,GO:0060038,GO:0060253,GO:0070374,GO:0071222,GO:0071456,GO:0071466,GO:0090307,GO:0090398,GO:1902895,GO:2000045,AA_TYPE_0,AA_TYPE_1,AA_TYPE_2,AA_TYPE_3,AA_TYPE_4,IS_PT_UKNOWN,is_frameshift,is_non_sens_mutation,IS_MISSENSE
str,f64,f64,f64,str,str,str,str,str,f64,f64,f64,f64,f64,f64,i32,f64,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,u32,u32,f64,i64,i64,i64,i64,i64,i64,…,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i32,i32,i32,i64
"""P100014""",3.0,7.9008e7,7.9009e7,null,null,"""KMT2A""","""MLL_PTD""","""PTD""",0.376635,355.550333,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null,1,1,0,0,0,0,…,0,0,0,0,0,1,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0
"""P100015""",3.0,7.9008e7,7.9009e7,null,null,"""KMT2A""","""MLL_PTD""","""PTD""",0.376635,355.550333,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null,1,1,0,0,0,0,…,0,0,0,0,0,1,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0
"""P100026""",3.0,7.9008e7,7.9009e7,null,null,"""KMT2A""","""MLL_PTD""","""PTD""",0.376635,355.550333,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null,1,1,0,0,0,0,…,0,0,0,0,0,1,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0
"""P100209""",3.0,7.9008e7,7.9009e7,null,null,"""KMT2A""","""MLL_PTD""","""PTD""",0.376635,355.550333,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null,1,1,0,0,0,0,…,0,0,0,0,0,1,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0
"""P100210""",3.0,7.9008e7,7.9009e7,null,null,"""KMT2A""","""MLL_PTD""","""PTD""",0.376635,355.550333,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null,1,1,0,0,0,0,…,0,0,0,0,0,1,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""P131472""",3.0,7.9008e7,7.9009e7,null,null,"""KMT2A""","""MLL_PTD""","""PTD""",0.376635,355.550333,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null,1,1,0,0,0,0,…,0,0,0,0,0,1,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0
"""P131505""",3.0,7.9008e7,7.9009e7,null,null,"""KMT2A""","""MLL_PTD""","""PTD""",0.376635,355.550333,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null,1,1,0,0,0,0,…,0,0,0,0,0,1,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0
"""P131816""",3.0,7.9008e7,7.9009e7,null,null,"""KMT2A""","""MLL_PTD""","""PTD""",0.376635,355.550333,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null,1,1,0,0,0,0,…,0,0,0,0,0,1,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0


# EFFECT

In [155]:
temp_df = mol_df.to_pandas()

encoder  = ce.BinaryEncoder(cols=["EFFECT"])

temp_df = encoder.fit_transform(temp_df)

temp_df

,ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT_0,EFFECT_1,...,GO:2000045,AA_TYPE_0,AA_TYPE_1,AA_TYPE_2,AA_TYPE_3,AA_TYPE_4,IS_PT_UKNOWN,is_frameshift,is_non_sens_mutation,IS_MISSENSE
0,P100000,11.0,1.191492e+08,1.191492e+08,G,A,CBL,p.C419Y,0,0,...,0,0,0,0,0,1,0,0,0,1
1,P100000,5.0,1.318223e+08,1.318223e+08,G,T,IRF1,p.Y164*,0,0,...,0,0,0,0,1,0,0,0,1,0
2,P100000,3.0,7.769406e+07,7.769406e+07,G,C,ROBO2,p.?,0,0,...,0,0,0,0,1,1,1,0,0,0
3,P100000,4.0,1.061649e+08,1.061649e+08,G,T,TET2,p.R1262L,0,0,...,0,0,0,1,0,0,0,0,0,1
4,P100000,2.0,2.546815e+07,2.546816e+07,ACGAAGAGGGGGTGTTC,A,DNMT3A,p.E505fs*141,0,1,...,0,0,0,1,0,1,0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10914,P131472,3.0,7.900754e+07,7.900930e+07,None,None,KMT2A,MLL_PTD,1,1,...,0,0,1,1,1,0,0,0,0,0
10915,P131505,3.0,7.900754e+07,7.900930e+07,None,None,KMT2A,MLL_PTD,1,1,...,0,0,1,1,1,0,0,0,0,0
10916,P131816,3.0,7.900754e+07,7.900930e+07,None,None,KMT2A,MLL_PTD,1,1,...,0,0,1,1,1,0,0,0,0,0
10917,P132717,3.0,7.900754e+07,7.900930e+07,None,None,KMT2A,MLL_PTD,1,1,...,0,0,1,1,1,0,0,0,0,0


# K-MER EXTRACT

In [156]:



mol_df = mol_df.with_columns(
    pl.when(pl.col("REF").str.len_chars() > k_mer_coef)
    .then(pl.col("REF").map_elements(lambda x: join_str(extract_kmers(x, k_mer_coef))))
    .otherwise(None)
    .alias(f"REF_{k_mer_coef}_mer")
)



C:\Users\zakar\AppData\Local\Temp\ipykernel_16716\2759431246.py:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  mol_df = mol_df.with_columns(


In [157]:
mol_df

ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH,is_complex_nucleotide,REF_length,ALT_length,is_transition,is_transversion,is_indel,REF_A,REF_C,REF_G,REF_T,ALT_A,ALT_C,ALT_G,ALT_T,complex_nucleotide_REF,complex_nucleotide_ALT,REF is_SNV,mutations_per_gene,mutations_per_gene_effect,mutations_per_region,REF_not_mer_0,REF_not_mer_1,REF_not_mer_2,REF_not_mer_3,REF_not_mer_4,REF_not_mer_5,…,GO:0045087,GO:0045747,GO:0045766,GO:0045892,GO:0045893,GO:0045944,GO:0046777,GO:0048511,GO:0048709,GO:0048873,GO:0050821,GO:0051276,GO:0051301,GO:0051321,GO:0051402,GO:0051726,GO:0051897,GO:0060038,GO:0060253,GO:0070374,GO:0071222,GO:0071456,GO:0071466,GO:0090307,GO:0090398,GO:1902895,GO:2000045,AA_TYPE_0,AA_TYPE_1,AA_TYPE_2,AA_TYPE_3,AA_TYPE_4,IS_PT_UKNOWN,is_frameshift,is_non_sens_mutation,IS_MISSENSE,REF_4_mer
str,f64,f64,f64,str,str,str,str,str,f64,f64,f64,f64,f64,f64,i32,f64,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,u32,u32,f64,i64,i64,i64,i64,i64,i64,…,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i32,i32,i32,i64,str
"""P100000""",11.0,1.19149248e8,1.19149248e8,"""G""","""A""","""CBL""","""p.C419Y""","""non_synonymous_codon""",0.083,1308.0,0.0,1.0,1.0,1.0,0,0.0,0,0,1,0,1,0,0,0,0,0,1,228,163,1.0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,null
"""P100000""",5.0,1.31822301e8,1.31822301e8,"""G""","""T""","""IRF1""","""p.Y164*""","""stop_gained""",0.022,532.0,0.0,1.0,1.0,0.0,1,0.0,0,0,1,0,0,0,0,1,0,0,1,22,4,1.0,0,0,0,0,0,0,…,1,0,0,1,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,null
"""P100000""",3.0,7.769406e7,7.769406e7,"""G""","""C""","""ROBO2""","""p.?""","""splice_site_variant""",0.41,876.0,0.0,1.0,1.0,0.0,1,0.0,0,0,1,0,0,1,0,0,0,0,1,17,3,3.0,0,0,0,0,0,0,…,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,null
"""P100000""",4.0,1.06164917e8,1.06164917e8,"""G""","""T""","""TET2""","""p.R1262L""","""non_synonymous_codon""",0.43,826.0,0.0,1.0,1.0,0.0,1,0.0,0,0,1,0,0,0,0,1,0,0,1,1663,417,4.0,0,0,0,0,0,0,…,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,null
"""P100000""",2.0,2.5468147e7,2.5468163e7,"""ACGAAGAGGGGGTGTTC""","""A""","""DNMT3A""","""p.E505fs*141""","""frameshift_variant""",0.0898,942.0,0.0,17.0,1.0,0.0,0,1.0,0,0,0,0,1,0,0,0,1,0,0,604,93,1.0,1,1,0,0,0,0,…,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,1,1,0,"""ACGA CGAA GAAG AAGA AGAG GAGG …"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""P131472""",3.0,7.9008e7,7.9009e7,null,null,"""KMT2A""","""MLL_PTD""","""PTD""",0.376635,355.550333,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null,1,1,0,0,0,0,…,0,0,0,0,1,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,null
"""P131505""",3.0,7.9008e7,7.9009e7,null,null,"""KMT2A""","""MLL_PTD""","""PTD""",0.376635,355.550333,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null,1,1,0,0,0,0,…,0,0,0,0,1,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,null
"""P131816""",3.0,7.9008e7,7.9009e7,null,null,"""KMT2A""","""MLL_PTD""","""PTD""",0.376635,355.550333,null,null,null,null,0,null,0,0,0,0,0,0,0,0,0,0,0,96,89,null,1,1,0,0,0,0,…,0,0,0,0,1,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,null


In [158]:
ref_4_mer = mol_df.filter(pl.col("REF_4_mer").is_not_null())["REF_4_mer"].to_list()

ref_4_mer

['ACGA CGAA GAAG AAGA AGAG GAGG AGGG GGGG GGGG GGGT GGTG GTGT TGTT GTTC',
 'TCAC CACC ACCA CCAC CACT ACTG CTGC TGCC GCCA CCAT CATA ATAG TAGA AGAG GAGA AGAG GAGG AGGC GGCG GCGG CGGC',
 'ACGC CGCG',
 'TCGC CGCG GCGG CGGC GGCC GCCG CCGT CGTC GTCC TCCA CCAG CAGC AGCA GCAC CACG ACGG',
 'TCAC CACC ACCA CCAC CACT ACTG CTGC TGCC GCCA CCAT CATA ATAG TAGA AGAG GAGA AGAG GAGG AGGC GGCG GCGG CGGC',
 'CAGA AGAT GATT ATTA TTAA TAAC AACC ACCA CCAG CAGG AGGA GGAC GACA',
 'TGTG GTGG TGGT GGTG GTGT TGTG GTGA TGAG GAGT AGTC GTCC TCCG CCGG CGGG GGGG GGGG GGGG GGGC GGCG GCGG CGGC GGCC',
 'TCAC CACC ACCA CCAC CACT ACTG CTGC TGCC GCCA CCAT CATA ATAG TAGA AGAG GAGA AGAG GAGG AGGC GGCG GCGG CGGC',
 'TGTG GTGG TGGT GGTG GTGT TGTG GTGA TGAG GAGT AGTC GTCC TCCG CCGG CGGG GGGG GGGG GGGG GGGC GGCG',
 'AAAA AAAG AAGT',
 'CTGT TGTT',
 'ACAG CAGT',
 'TCAC CACC ACCA CCAC CACT ACTG CTGC TGCC GCCA CCAT CATA ATAG TAGA AGAG GAGA AGAG GAGG AGGC GGCG GCGG CGGC',
 'CGCA GCAG CAGT AGTT',
 'TCAC CACC ACCA CCAC CACT ACTG CTGC TG

In [159]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(ngram_range=(4,4))


X_ref_4_mer = cv.fit_transform(ref_4_mer)

X_ref_4_mer



<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 6271 stored elements and shape (558, 1926)>

In [160]:
feature_names_4mer = cv.get_feature_names_out()

# Convertir la matrice sparse en DataFrame
ref_4mer_df = pd.DataFrame(
    X_ref_4_mer.toarray(),
    columns=feature_names_4mer
)

ref_4mer_df



,aaaa aaaa aaac aact,aaaa aaaa aaat aatt,aaaa aaac aaca acag,aaaa aaac aact actg,aaaa aaac aact actt,aaaa aaag aagc agct,aaaa aaag aagt agtg,aaaa aaat aata atag,aaaa aaat aatt attt,aaac aaca acag cagc,...,tttg ttga tgaa gaag,tttg ttga tgac gacc,tttg ttga tgat gatg,tttg ttgc tgct gctg,tttg ttgc tgct gctt,tttg ttgg tggt ggta,tttg ttgg tggt ggtg,tttt tttc ttca tcat,tttt tttg ttgc tgct,tttt tttt tttc ttca
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
553,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
554,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
555,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
556,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
